# OBT modeling

This Colab notebook was generated from the FeatureMesh docs tutorial.

Work top to bottom: first install the FeatureMesh client, then create clients, then load data and run the tutorial.

1. **Install FeatureMesh** — Python packages for this Colab runtime.
2. **Create the BatchClient** — Jupyter magic and a local DuckDB-backed client.
3. **Follow the tutorial** — run the remaining cells in order.


## 1. Install FeatureMesh

Install the client packages for this Colab runtime.


In [ ]:
%pip install -q featuremesh pandas


## 2. Create the BatchClient

Load the Jupyter magic and create a local `BatchClient`.


In [ ]:
%load_ext featuremesh


In [ ]:
from IPython.display import display
from featuremesh import BatchClient, set_default

client = BatchClient()
set_default("client", client)
print("FeatureMesh BatchClient ready (local DuckDB)")


Work an entity-centric one-big-table: each account holds nested contacts, opportunities (with line items), and activities. Filter and aggregate inside those arrays, pick the top deal and line item, derive a lifecycle stage, and pack the metrics into one profile `ROW` — without fan-out joins across arrays.

This advanced tutorial assumes entity bindings from [E-commerce](https://featuremesh.com/docs/tutorials/analytics/ecomm). Read each `TRANSFORM(SELECT …)` as a SQL query scoped to one account's nested array. Run Data and Model before the analysis, and treat the profile section as the capstone.


## Data

Three B2B accounts in one denormalized table. **Alpha** has a won deal and an open upsell. **Beta** is busy in March but has not closed. **Delta** signed long ago, has no decision-maker, and has gone quiet.


In [3]:
%%featureql --client client --hide-dataframe

DROP FEATURES IF EXISTS IN FM.OBT UP TO LEVEL 9;


*** Warning(s) (1) ***

  1. [DROP-FEATURES-EXPRESSION-EMPTY] No features found in expression: *(FEATURES IF EXISTS IN FM.OBT UP TO LEVEL 9) (acknowledge with ACK-4YHD)


In [4]:
%%featureql --client client --hide-dataframe

/* SQL */
CREATE SCHEMA IF NOT EXISTS tutorial_obt;
--
DROP TABLE IF EXISTS tutorial_obt.accounts;
--
CREATE TABLE tutorial_obt.accounts (
    id BIGINT,
    name VARCHAR,
    contacts STRUCT(name VARCHAR, role VARCHAR)[],
    opportunities STRUCT(
        opp_id BIGINT,
        name VARCHAR,
        amount BIGINT,
        stage VARCHAR,
        line_items STRUCT(item_name VARCHAR, amount BIGINT)[]
    )[],
    activities STRUCT(
        activity_id BIGINT,
        activity_type VARCHAR,
        ts TIMESTAMP
    )[]
);
--
INSERT INTO tutorial_obt.accounts VALUES
(
    1, 'Alpha',
    [
        {name: 'Alice', role: 'champion'},
        {name: 'Bob', role: 'decision_maker'}
    ],
    [
        {opp_id: 101, name: 'Alpha Expansion', amount: 50000, stage: 'closed_won',
         line_items: [
            {item_name: 'Platform License', amount: 20000},
            {item_name: 'Support', amount: 15000},
            {item_name: 'Training', amount: 15000}
         ]},
        {opp_id: 102, name: 'Alpha Upsell', amount: 120000, stage: 'negotiation',
         line_items: [
            {item_name: 'Enterprise License', amount: 80000},
            {item_name: 'Data Add-on', amount: 40000}
         ]}
    ],
    [
        {activity_id: 1, activity_type: 'demo', ts: TIMESTAMP '2024-01-15 14:00:00'},
        {activity_id: 2, activity_type: 'contract_signed', ts: TIMESTAMP '2024-01-20 10:00:00'},
        {activity_id: 3, activity_type: 'call', ts: TIMESTAMP '2024-03-01 15:00:00'},
        {activity_id: 4, activity_type: 'demo', ts: TIMESTAMP '2024-03-15 14:00:00'},
        {activity_id: 5, activity_type: 'page_visit', ts: TIMESTAMP '2024-03-28 16:00:00'}
    ]
),
(
    2, 'Beta',
    [
        {name: 'Dan', role: 'decision_maker'},
        {name: 'Eve', role: 'champion'}
    ],
    [
        {opp_id: 201, name: 'Beta Initial', amount: 30000, stage: 'proposal',
         line_items: [
            {item_name: 'Platform License', amount: 20000},
            {item_name: 'Onboarding', amount: 10000}
         ]}
    ],
    [
        {activity_id: 11, activity_type: 'demo', ts: TIMESTAMP '2024-02-10 14:00:00'},
        {activity_id: 12, activity_type: 'call', ts: TIMESTAMP '2024-03-05 11:00:00'},
        {activity_id: 13, activity_type: 'email_open', ts: TIMESTAMP '2024-03-08 09:00:00'},
        {activity_id: 14, activity_type: 'call', ts: TIMESTAMP '2024-03-12 14:00:00'},
        {activity_id: 15, activity_type: 'page_visit', ts: TIMESTAMP '2024-03-18 10:00:00'},
        {activity_id: 16, activity_type: 'demo', ts: TIMESTAMP '2024-03-20 15:00:00'},
        {activity_id: 17, activity_type: 'call', ts: TIMESTAMP '2024-03-25 16:00:00'}
    ]
),
(
    3, 'Delta',
    [
        {name: 'Fay', role: 'champion'}
    ],
    [
        {opp_id: 301, name: 'Delta Pilot', amount: 10000, stage: 'closed_won',
         line_items: [
            {item_name: 'Starter License', amount: 8000},
            {item_name: 'Setup', amount: 2000}
         ]}
    ],
    [
        {activity_id: 21, activity_type: 'contract_signed', ts: TIMESTAMP '2024-02-01 12:00:00'}
    ]
);


In [5]:
%%featureql --client client

/* SQL */
SELECT id, name
FROM tutorial_obt.accounts
ORDER BY id;


,id,name
0,1,Alpha
1,2,Beta
2,3,Delta


## Model

Bind `ACCOUNT_ID`. Nested columns become typed `ARRAY(ROW(...))` features you can `TRANSFORM` without unnesting into the outer query.


In [6]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.OBT AS
SELECT
    accounts := ENTITY(),
    account_id := INPUT(BIGINT#accounts)
;


,feature_name,status,message
0,FM.OBT.ACCOUNTS,CREATED,Feature created as not exists
1,FM.OBT.ACCOUNT_ID,CREATED,Feature created as not exists


In [7]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.OBT AS
SELECT
    tables.accounts := EXTERNAL_COLUMNS(
        id BIGINT#accounts BIND TO account_id,
        name VARCHAR,
        contacts ARRAY(ROW(name VARCHAR, role VARCHAR)),
        opportunities ARRAY(
            ROW(
                opp_id BIGINT,
                name VARCHAR,
                amount BIGINT,
                stage VARCHAR,
                line_items ARRAY(ROW(item_name VARCHAR, amount BIGINT))
            )
        ),
        activities ARRAY(
            ROW(activity_id BIGINT, activity_type VARCHAR, ts TIMESTAMP)
        )
        FROM SQL(
            SELECT id, name, contacts, opportunities, activities
            FROM tutorial_obt.accounts
        )
    ),
    account_name := tables.accounts[name],
    contacts := tables.accounts[contacts],
    opportunities := tables.accounts[opportunities],
    activities := tables.accounts[activities]
;


,feature_name,status,message
0,FM.OBT.TABLES.ACCOUNTS,CREATED,Feature created as not exists
1,FM.OBT.ACCOUNT_NAME,CREATED,Feature created as not exists
2,FM.OBT.CONTACTS,CREATED,Feature created as not exists
3,FM.OBT.OPPORTUNITIES,CREATED,Feature created as not exists
4,FM.OBT.ACTIVITIES,CREATED,Feature created as not exists


## Count inside an array

Decision-makers live in `contacts[]`. Aggregate **inside** the array; unwrap the scalar.


In [8]:
%%featureql --client client

WITH
    DM_COUNT := COALESCE(
        CONTACTS.TRANSFORM(SELECT COUNT(*) FILTER (WHERE role = 'decision_maker')).UNWRAP_ONE(),
        0::BIGINT
    ),
SELECT
    ACCOUNT_NAME,
    DM_COUNT
FROM FM.OBT
FOR
    ACCOUNT_ID := BIND_VALUES(ARRAY[1, 2, 3])
ORDER BY ACCOUNT_NAME;


,FM.OBT.ACCOUNT_NAME,DM_COUNT
0,Alpha,1
1,Beta,1
2,Delta,0


Alpha **1**, Beta **1**, Delta **0**.

## Argmax: last activity

`ORDER BY … LIMIT 1` inside `TRANSFORM` returns the whole activity row — not just `MAX(ts)`.


In [9]:
%%featureql --client client

WITH
    LAST_ACT := ACTIVITIES.TRANSFORM(
        SELECT activity_type, ts ORDER BY ts DESC, activity_id DESC LIMIT 1
    ),
    ACTIVITY_TYPE := LAST_ACT.TRANSFORM(SELECT activity_type).UNWRAP_ONE(),
    ACTIVITY_DATE := CAST(LAST_ACT.TRANSFORM(SELECT ts).UNWRAP_ONE() AS DATE),
    DAYS_SINCE := DATE_DIFF(DATE '2024-04-01', ACTIVITY_DATE, 'day'),
SELECT
    ACCOUNT_NAME,
    ACTIVITY_TYPE,
    ACTIVITY_DATE,
    DAYS_SINCE
FROM FM.OBT
FOR
    ACCOUNT_ID := BIND_VALUES(ARRAY[1, 2, 3])
ORDER BY ACCOUNT_NAME;


,FM.OBT.ACCOUNT_NAME,ACTIVITY_TYPE,ACTIVITY_DATE,DAYS_SINCE
0,Alpha,page_visit,2024-03-28,4
1,Beta,call,2024-03-25,7
2,Delta,contract_signed,2024-02-01,60


As of 2024-04-01: Alpha **4** days (page_visit), Beta **7** (call), Delta **60** (contract_signed).

## Two arrays, no cross product

Sum opportunity amounts and count activities as **separate** array transforms. Dual `UNNEST` would explode; this does not.


In [10]:
%%featureql --client client

WITH
    OPP_VALUE := COALESCE(OPPORTUNITIES.TRANSFORM(SELECT SUM(amount)).UNWRAP_ONE(), 0::BIGINT),
    ACTIVITY_COUNT := COALESCE(ACTIVITIES.TRANSFORM(SELECT COUNT(*)).UNWRAP_ONE(), 0::BIGINT),
SELECT
    ACCOUNT_NAME,
    OPP_VALUE,
    ACTIVITY_COUNT
FROM FM.OBT
FOR
    ACCOUNT_ID := BIND_VALUES(ARRAY[1, 2, 3])
ORDER BY OPP_VALUE DESC, ACCOUNT_NAME;


,FM.OBT.ACCOUNT_NAME,OPP_VALUE,ACTIVITY_COUNT
0,Alpha,170000,5
1,Beta,30000,7
2,Delta,10000,1


Alpha **170000 / 5**, Beta **30000 / 7**, Delta **10000 / 1**.

## Engaged but not buying

Recent activity count (array predicate) **and** no `closed_won` (other array). Anded at account grain.


In [11]:
%%featureql --client client

WITH
    RECENT_COUNT := COALESCE(
        ACTIVITIES.TRANSFORM(
            SELECT COUNT(*) FILTER (
                WHERE ts >= TIMESTAMP '2024-03-02 00:00:00'
                  AND ts < TIMESTAMP '2024-04-01 00:00:00'
            )
        ).UNWRAP_ONE(),
        0::BIGINT
    ),
    HAS_CLOSED_WON := COALESCE(
        OPPORTUNITIES.TRANSFORM(SELECT BOOL_OR(stage = 'closed_won')).UNWRAP_ONE(),
        FALSE
    ),
SELECT
    ACCOUNT_NAME,
    RECENT_COUNT,
    HAS_CLOSED_WON
FROM FM.OBT
FOR
    ACCOUNT_ID := BIND_VALUES(ARRAY[1, 2, 3])
WHERE RECENT_COUNT > 5 AND NOT HAS_CLOSED_WON
ORDER BY ACCOUNT_NAME;


,FM.OBT.ACCOUNT_NAME,RECENT_COUNT,HAS_CLOSED_WON
0,Beta,6,False


**Beta** only (6 March activities, open proposal).

## Nested arrays: top deal + top line item

Argmax on opportunities by sum of `line_items[]`, then argmax again inside that opportunity’s items.


In [12]:
%%featureql --client client

WITH
    TOP_OPP := OPPORTUNITIES.TRANSFORM(
        WITH items_total := COALESCE(ARRAY_SUM(line_items[amount]), 0::BIGINT)
        SELECT name, items_total, line_items
        ORDER BY items_total DESC, opp_id ASC
        LIMIT 1
    ),
    OPP_NAME := TOP_OPP.TRANSFORM(SELECT name).UNWRAP_ONE(),
    OPP_TOTAL := TOP_OPP.TRANSFORM(SELECT items_total).UNWRAP_ONE(),
    TOP_ITEM_ROW := TOP_OPP.TRANSFORM(SELECT line_items).UNWRAP_ONE().TRANSFORM(
        SELECT item_name, amount ORDER BY amount DESC, item_name ASC LIMIT 1
    ),
    TOP_ITEM := TOP_ITEM_ROW.TRANSFORM(SELECT item_name).UNWRAP_ONE(),
    TOP_ITEM_AMT := TOP_ITEM_ROW.TRANSFORM(SELECT amount).UNWRAP_ONE(),
SELECT
    ACCOUNT_NAME,
    OPP_NAME,
    OPP_TOTAL,
    TOP_ITEM,
    TOP_ITEM_AMT
FROM FM.OBT
FOR
    ACCOUNT_ID := BIND_VALUES(ARRAY[1, 2, 3])
ORDER BY ACCOUNT_NAME;


,FM.OBT.ACCOUNT_NAME,OPP_NAME,OPP_TOTAL,TOP_ITEM,TOP_ITEM_AMT
0,Alpha,Alpha Upsell,120000,Enterprise License,80000
1,Beta,Beta Initial,30000,Platform License,20000
2,Delta,Delta Pilot,10000,Starter License,8000


Alpha → **Alpha Upsell** / **120000** / **Enterprise License** / **80000**.

## Lifecycle from the activity array

Latest type drives `customer`; else demo-without-contract → `prospect`; else `other`. Historical `contract_signed` does **not** make you a customer if something newer happened.


In [13]:
%%featureql --client client

WITH
    LAST_ACTIVITY_TYPE := ACTIVITIES.TRANSFORM(
        SELECT activity_type ORDER BY ts DESC, activity_id DESC LIMIT 1
    ).UNWRAP_ONE(),
    HAS_DEMO := COALESCE(ACTIVITIES.TRANSFORM(SELECT BOOL_OR(activity_type = 'demo')).UNWRAP_ONE(), FALSE),
    HAS_CONTRACT := COALESCE(ACTIVITIES.TRANSFORM(SELECT BOOL_OR(activity_type = 'contract_signed')).UNWRAP_ONE(), FALSE),
    LIFECYCLE_STAGE := CASE
        WHEN LAST_ACTIVITY_TYPE = 'contract_signed' THEN 'customer'
        WHEN HAS_DEMO AND NOT HAS_CONTRACT THEN 'prospect'
        ELSE 'other'
    END,
SELECT
    ACCOUNT_NAME,
    LIFECYCLE_STAGE
FROM FM.OBT
FOR
    ACCOUNT_ID := BIND_VALUES(ARRAY[1, 2, 3])
ORDER BY ACCOUNT_NAME;


,FM.OBT.ACCOUNT_NAME,LIFECYCLE_STAGE
0,Alpha,other
1,Beta,prospect
2,Delta,customer


Alpha **other**, Beta **prospect**, Delta **customer**.

## Re-nest a profile

Fold the metrics into one `ROW` per account, plus a variable-length `risk_flags` array.


In [14]:
%%featureql --client client

WITH
    decision_maker_count := COALESCE(
        contacts.TRANSFORM(
            SELECT COUNT(*) FILTER (WHERE role = 'decision_maker')
        ).UNWRAP_ONE(),
        0::BIGINT
    ),
    last_act := activities.TRANSFORM(
        SELECT ACTIVITY_TYPE, TS ORDER BY TS DESC, ACTIVITY_ID DESC LIMIT 1
    ),
    last_activity_type := last_act.TRANSFORM(SELECT ACTIVITY_TYPE).UNWRAP_ONE(),
    days_since_last_activity := DATE_DIFF(
        DATE '2024-04-01',
        CAST(last_act.TRANSFORM(SELECT TS).UNWRAP_ONE() AS DATE),
        'day'
    ),
    total_opportunity_value := COALESCE(
        opportunities.TRANSFORM(SELECT SUM(amount)).UNWRAP_ONE(),
        0::BIGINT
    ),
    has_demo := COALESCE(
        activities.TRANSFORM(SELECT BOOL_OR(activity_type = 'demo')).UNWRAP_ONE(),
        FALSE
    ),
    has_contract := COALESCE(
        activities.TRANSFORM(SELECT BOOL_OR(activity_type = 'contract_signed')).UNWRAP_ONE(),
        FALSE
    ),
    lifecycle_stage := CASE
        WHEN last_activity_type = 'contract_signed' THEN 'customer'
        WHEN has_demo AND NOT has_contract THEN 'prospect'
        ELSE 'other'
    END,
    risk_flags := ARRAY(
        IF(
            decision_maker_count = 0,
            ROW('no_decision_maker' AS flag),
            NULL(ROW(flag VARCHAR))
        ),
        IF(
            days_since_last_activity > 30,
            ROW('gone_dark' AS flag),
            NULL(ROW(flag VARCHAR))
        ),
        IF(
            total_opportunity_value = 0,
            ROW('no_pipeline' AS flag),
            NULL(ROW(flag VARCHAR))
        )
    ).TRANSFORM(SELECT * WHERE FLAG IS NOT NULL),
    profile := ROW(
        account_name AS account_name,
        decision_maker_count AS decision_maker_count,
        days_since_last_activity AS days_since_last_activity,
        total_opportunity_value AS total_opportunity_value,
        lifecycle_stage AS lifecycle_stage,
        risk_flags AS risk_flags
    )
SELECT
    profile
FROM FM.OBT
FOR
    account_id := BIND_VALUES(ARRAY(1, 2, 3))
ORDER BY profile[account_name]
;


,PROFILE
0,"{'account_name': 'Alpha', 'decision_maker_coun..."
1,"{'account_name': 'Beta', 'decision_maker_count..."
2,"{'account_name': 'Delta', 'decision_maker_coun..."


Delta carries **no_decision_maker** and **gone_dark**. Alpha and Beta have empty flags.

## What's next

- [Analytics overview](https://featuremesh.com/docs/tutorials/analytics/overview) — full series map
- [Graphs & referrals](https://featuremesh.com/docs/tutorials/analytics/graph) — `RECURSE()` over parent links
- [Marketing attribution](https://featuremesh.com/docs/tutorials/analytics/marketing) — `TRANSFORM` over journey arrays
- [E-commerce](https://featuremesh.com/docs/tutorials/analytics/ecomm) — relationship shapes that feed nested models


---

Source tutorial: [/docs/tutorials/analytics/obt](https://featuremesh.com/docs/tutorials/analytics/obt)
